In [2]:
import sys
import os

if not os.path.exists("config.py"):
    os.chdir("backend") if os.path.exists("backend") else os.chdir("..")

sys.path.insert(0, os.getcwd())

print("Working directory:", os.getcwd())
print("config.py exists:", os.path.exists("config.py"))

Working directory: c:\Users\Dell\Documents\repo\llms\document-assistant\backend
config.py exists: True


In [4]:
import io
import re
import pdfplumber
from openai import OpenAI
from supabase import create_client
from googleapiclient.discovery import build
from google.oauth2.credentials import Credentials
from googleapiclient.http import MediaIoBaseDownload
from config import settings

openai_client = OpenAI(api_key=settings.openai_api_key)
sb = create_client(settings.supabase_url, settings.supabase_service_role_key)

SCOPES = ["https://www.googleapis.com/auth/drive.readonly"]
creds = Credentials.from_authorized_user_file(settings.google_token_file, SCOPES)
service = build("drive", "v3", credentials=creds)

print("Clients ready")

Clients ready


In [5]:
PDF_FILE_ID = "1c-18lHMiW-wxFnUlhd6y21RNlP6ex_Ek"

def download_file(file_id: str) -> bytes:
    request = service.files().get_media(fileId=file_id)
    buffer = io.BytesIO()
    downloader = MediaIoBaseDownload(buffer, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()
    return buffer.getvalue()

def clean_text(text: str) -> str:
    text = re.sub(r'\.{4,}\s*\d+', '', text)
    text = re.sub(r' {2,}', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def chunk_by_clauses(text: str, max_tokens: int = 1500) -> list[dict]:
    text = clean_text(text)
    pattern = re.compile(r'(?m)^(\d+\.\d+(?:\.\d+)?)\s+(.+)')
    matches = list(pattern.finditer(text))
    chunks = []
    for i, match in enumerate(matches):
        clause_ref = match.group(1)
        clause_title = match.group(2).strip()
        start = match.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        content = text[start:end].strip()
        if len(content) < 50:
            continue
        words = content.split()
        if len(words) > max_tokens:
            step = max_tokens
            overlap = 100
            for j in range(0, len(words), step - overlap):
                sub_content = " ".join(words[j:j + step])
                chunks.append({
                    "clause_ref": f"{clause_ref} (part {j // (step - overlap) + 1})",
                    "clause_title": clause_title,
                    "content": sub_content,
                    "char_start": start,
                    "char_end": end,
                })
        else:
            chunks.append({
                "clause_ref": clause_ref,
                "clause_title": clause_title,
                "content": content,
                "char_start": start,
                "char_end": end,
            })
    return chunks

pdf_bytes = download_file(PDF_FILE_ID)

with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
    full_text = "\n".join(
        page.extract_text() for page in pdf.pages[8:]
        if page.extract_text()
    )

chunks = chunk_by_clauses(full_text)
print(f"Chunks ready: {len(chunks)}")

Chunks ready: 773


In [7]:
def embed_text(text: str) -> list[float]:
    response = openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

# Test on first chunk
test_chunk = chunks[0]
embedding = embed_text(test_chunk["content"])

print(f"Chunk: {test_chunk['clause_ref']}")
print(f"Embedding length: {len(embedding)}")
print(f"First 5 values: {embedding[:5]}")

Chunk: 1.1.1
Embedding length: 1536
First 5 values: [-0.00891876220703125, -0.0290069580078125, 0.04388427734375, -0.013336181640625, -0.0288238525390625]


In [6]:
print("Key length:", len(settings.openai_api_key))

Key length: 164


In [8]:
import uuid

# Use a fake file_id for now — we'll wire up real file tracking in the module
test_file_id = str(uuid.uuid4())

row = {
    "file_id": test_file_id,
    "project": "QMC Heritage",
    "filename": "Contract C2024-49.pdf",
    "clause_ref": test_chunk["clause_ref"],
    "content": test_chunk["content"],
    "char_start": test_chunk["char_start"],
    "char_end": test_chunk["char_end"],
    "embedding": embedding,
}

result = sb.table("contract_chunks").insert(row).execute()
print("Inserted:", result.data[0]["id"])

Inserted: 25f97e1a-e78f-45f4-a207-494e3db789cd


In [9]:
import time

BATCH_SIZE = 100
PROJECT = "QMC Heritage"
FILENAME = "Contract C2024-49.pdf"
FILE_ID = str(uuid.uuid4())  # temporary — will be real Drive file ID when wired into module

def embed_batch(texts: list[str]) -> list[list[float]]:
    response = openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )
    return [item.embedding for item in response.data]

rows = []
for i in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[i:i + BATCH_SIZE]
    texts = [c["content"] for c in batch]
    embeddings = embed_batch(texts)
    
    for chunk, embedding in zip(batch, embeddings):
        rows.append({
            "file_id": FILE_ID,
            "project": PROJECT,
            "filename": FILENAME,
            "clause_ref": chunk["clause_ref"],
            "content": chunk["content"],
            "char_start": chunk["char_start"],
            "char_end": chunk["char_end"],
            "embedding": embedding,
        })
    
    print(f"Embedded {min(i + BATCH_SIZE, len(chunks))}/{len(chunks)} chunks")
    time.sleep(0.5)  # avoid rate limiting

print(f"\nTotal rows ready to insert: {len(rows)}")

Embedded 100/773 chunks
Embedded 200/773 chunks
Embedded 300/773 chunks
Embedded 400/773 chunks
Embedded 500/773 chunks
Embedded 600/773 chunks
Embedded 700/773 chunks
Embedded 773/773 chunks

Total rows ready to insert: 773


In [10]:
# Insert in batches of 100
for i in range(0, len(rows), BATCH_SIZE):
    batch = rows[i:i + BATCH_SIZE]
    sb.table("contract_chunks").insert(batch).execute()
    print(f"Inserted {min(i + BATCH_SIZE, len(rows))}/{len(rows)} rows")

print("\nDone — all chunks in Supabase ✓")

Inserted 100/773 rows
Inserted 200/773 rows
Inserted 300/773 rows
Inserted 400/773 rows
Inserted 500/773 rows
Inserted 600/773 rows
Inserted 700/773 rows
Inserted 773/773 rows

Done — all chunks in Supabase ✓


In [11]:
result = sb.table("contract_chunks").select("clause_ref, content, filename, project").limit(5).execute()
for row in result.data:
    print(f"Clause: {row['clause_ref']}")
    print(f"Project: {row['project']}")
    print(f"Content: {row['content'][:100]}")
    print()

count = sb.table("contract_chunks").select("id", count="exact").execute()
print(f"Total rows in Supabase: {count.count}")

Clause: 1.1.1
Project: QMC Heritage
Content: 1.1.1 The defined words and expressions set out in Clause 1 of Appendix 1 [Definitions and
Interpret

Clause: 1.1.1
Project: QMC Heritage
Content: 1.1.1 The defined words and expressions set out in Clause 1 of Appendix 1 [Definitions and
Interpret

Clause: 1.2.1
Project: QMC Heritage
Content: 1.2.1 The following documents constitute the Contract Documents and shall be taken as
mutually expla

Clause: 1.2.2
Project: QMC Heritage
Content: 1.2.2 In the event that there exists a conflict, ambiguity or discrepancy within the Contract
Docume

Clause: 1.3.1
Project: QMC Heritage
Content: 1.3.1 The Contract shall come into effect on the Effective Date. Notwithstanding the
effectiveness o

Total rows in Supabase: 774


In [12]:
sb.table("contract_chunks").delete().eq("file_id", FILE_ID).limit(1).execute()

count = sb.table("contract_chunks").select("id", count="exact").execute()
print(f"Total rows after cleanup: {count.count}")

AttributeError: 'SyncFilterRequestBuilder' object has no attribute 'limit'

In [13]:
result = sb.table("contract_chunks").select("file_id").limit(5).execute()
for row in result.data:
    print(row["file_id"])

21ee8f37-89e9-491a-8f2e-3c609a56784f
eabeed3e-5cce-4590-b762-aa4a70ed54ff
eabeed3e-5cce-4590-b762-aa4a70ed54ff
eabeed3e-5cce-4590-b762-aa4a70ed54ff
eabeed3e-5cce-4590-b762-aa4a70ed54ff


In [14]:
sb.table("contract_chunks").delete().eq("file_id", "21ee8f37-89e9-491a-8f2e-3c609a56784f").execute()

count = sb.table("contract_chunks").select("id", count="exact").execute()
print(f"Total rows after cleanup: {count.count}")

Total rows after cleanup: 773
